# 01. 학습 설정 파일 해부 (Config Anatomy)

이 노트북은 `DRL_training_config/` 폴더의 50개 설정 파일이 **무엇을 기록하고 있는지** 읽어 봅니다.

설정 파일은 비공개 코드베이스(`agent.*`, DI-engine 학습 파이프라인)를 import 하므로 그대로 실행하면 `ModuleNotFoundError`가 납니다.
여기서는 그 import를 **가짜 모듈(stub)로 대체**하여 파일을 로드하고, 그 안에 정의된 `main_config` / `create_config` 딕셔너리만 꺼내 봅니다.

> 실행 커널: `Python (ppo_rl)` — `bash tutorial_nbs/setup_env.sh` 로 만든 conda 환경입니다.

In [1]:
import sys, os, types, runpy, glob, warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "tutorial_nbs" else Path.cwd()
CFG_DIR = ROOT / "DRL_training_config"
print("config dir:", CFG_DIR, "| files:", len(list(CFG_DIR.glob("*_config.py"))))

config dir: /home/hydro/research/rl/PPO-Lagrangian-Paper/DRL_training_config | files: 50


## 1. 설정 파일을 안전하게 로드하기

파일 상단의 `from agent.tool_function import ppo_lagrangian` 은 PPO-Lagrangian 정책을 DI-engine 레지스트리에 등록하는 줄입니다.
비공개 패키지이므로 `sys.modules`에 빈 모듈을 넣어 통과시킵니다. 그 외에는 `easydict`, `pytz`, `torch`, `gym` 만 있으면 됩니다.

In [2]:
def load_config(path):
    """Execute a *_config.py with the private `agent` package stubbed out and return its namespace."""
    for name in ("agent", "agent.tool_function", "agent.tool_function.ppo_lagrangian"):
        sys.modules.setdefault(name, types.ModuleType(name))
    sys.modules["agent.tool_function"].ppo_lagrangian = sys.modules["agent.tool_function.ppo_lagrangian"]
    ns = runpy.run_path(str(path), run_name="__tutorial__")   # run_name != "__main__" -> training is NOT launched
    return ns

ns = load_config(CFG_DIR / "A3_Lagrangian_R1_config.py")
main_config, create_config = ns["main_config"], ns["create_config"]
print(type(main_config).__name__, "| policy type:", create_config.policy.type, "| exp_name:", main_config.exp_name)

EasyDict | policy type: ppo_lagrangian | exp_name: agent/reservoir_single_agent/DRL_training_config/A3_Lagrangian_R1_202511180027


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## 2. `main_config` 들여다보기

DI-engine 설정은 `env` 와 `policy` 두 블록으로 나뉩니다. 논문 본문에 보고된 하이퍼파라미터가 어디에 기록되어 있는지 확인합니다.

In [3]:
def flat(d, prefix=""):
    for k, v in d.items():
        if isinstance(v, dict):
            yield from flat(v, f"{prefix}{k}.")
        else:
            yield f"{prefix}{k}", v

env_tbl = pd.DataFrame(list(flat(main_config.env)), columns=["key", "value"])
env_tbl[~env_tbl.key.str.startswith("tensorboard")]

,key,value
0,episode_num,60
1,collector_env_num,5
2,evaluator_env_num,1
3,env_id,ResevoirAgent-v0
4,act_scale,False
5,n_evaluator_episode,1
6,stop_value,100000
7,continuous,False
8,action_space,discrete
9,action_shape,100


In [4]:
pol = main_config.policy
pd.DataFrame(list(flat({"model": pol.model, "learn": {k: v for k, v in pol.learn.items() if k != "learner"},
                        "collect": {k: v for k, v in pol.collect.items() if k != "collector"},
                        "lagrangian": pol.lagrangian})), columns=["key", "value"])

,key,value
0,model.obs_shape,6
1,model.action_shape,100
2,model.encoder_hidden_size_list,"[512, 512, 256]"
3,model.action_space,discrete
4,model.actor_head_layer_num,1
5,model.critic_head_layer_num,1
6,model.actor_head_hidden_size,256
7,model.critic_head_hidden_size,256
8,learn.debug_print_entropy,False
9,learn.resume_training,False


### 파생 값: 총 에피소드 수와 학습 반복 횟수

README 에 설명된 대로 `episode_num` 은 **collector 환경당** 값입니다. 파일 안의 상수로 논문의 수치를 재계산해 봅니다.

In [5]:
CO, EP, STEP, BS, UPD = ns["CO_NUM"], ns["MAX_EPISODE"], ns["MAX_EPISODE_STEP"], ns["BATCH_SIZE"], ns["UPDATE_NUM"]
print(f"collector envs           : {CO}")
print(f"episodes per env         : {EP}   -> total episodes = {CO*EP}")
print(f"steps per episode        : {STEP} (= 365 days x 4 years)")
print(f"n_sample per collect     : {ns['N_SAMPLE']}  (= STEP/2 * CO)")
print(f"batch_size               : {BS}, epochs per collect: {UPD}")
print(f"gradient steps (TOTAL_STEPS): {ns['TOTAL_STEPS']}")
print(f"learner train_iterations : {pol.learn.learner.train_iterations}")
print(f"log every                : {ns['LOG_SHOW_AFTER_ITER']} iters, ckpt every {pol.learn.learner.hook.save_ckpt_after_iter} iters")

collector envs           : 5
episodes per env         : 60   -> total episodes = 300
steps per episode        : 1460 (= 365 days x 4 years)
n_sample per collect     : 3650  (= STEP/2 * CO)
batch_size               : 1460, epochs per collect: 10
gradient steps (TOTAL_STEPS): 3000
learner train_iterations : 3050
log every                : 30 iters, ckpt every 750 iters


## 3. 50개 파일의 스위치를 한 표로

모든 파일을 로드해서 실험 스위치만 뽑아 봅니다. 같은 변형(variant)의 R1–R5 는 `filename_without_ext` 만 다르다는 것을 확인할 수 있습니다.

In [6]:
SWITCHES = ["USE_PPO_LAGRANGIAN", "CONSTRAINT_TYPES", "ENABLE_ACTION_MASK", "MASK_RULE_TYPE",
            "ENABLE_NONLINEAR_MAPPING", "NONLINEAR_MAPPING_TYPE", "POLICY_TYPE"]
rows = []
for p in sorted(CFG_DIR.glob("*_config.py")):
    n = load_config(p)
    variant, run = p.stem.rsplit("_R", 1)
    rows.append(dict(variant=variant, run=f"R{run.split('_')[0]}", run_name=n["filename_without_ext"], **{k: n[k] for k in SWITCHES}))
df = pd.DataFrame(rows)
df.head(10)

,variant,run,run_name,USE_PPO_LAGRANGIAN,CONSTRAINT_TYPES,ENABLE_ACTION_MASK,MASK_RULE_TYPE,ENABLE_NONLINEAR_MAPPING,NONLINEAR_MAPPING_TYPE,POLICY_TYPE
0,A1_Correction,R1,A1_Correction_R1_202511172159,False,2,False,basic,False,square,ppo
1,A1_Correction,R2,A1_Correction_R2_202511172218,False,2,False,basic,False,square,ppo
2,A1_Correction,R3,A1_Correction_R3_202511172232,False,2,False,basic,False,square,ppo
3,A1_Correction,R4,A1_Correction_R4_202511172246,False,2,False,basic,False,square,ppo
4,A1_Correction,R5,A1_Correction_R5_202511172301,False,2,False,basic,False,square,ppo
5,A2_Penalty,R1,A2_Penalty_R1_202511172315,False,3,False,basic,False,square,ppo
6,A2_Penalty,R2,A2_Penalty_R2_202511172330,False,3,False,basic,False,square,ppo
7,A2_Penalty,R3,A2_Penalty_R3_202511172344,False,3,False,basic,False,square,ppo
8,A2_Penalty,R4,A2_Penalty_R4_202511172358,False,3,False,basic,False,square,ppo
9,A2_Penalty,R5,A2_Penalty_R5_202511180013,False,3,False,basic,False,square,ppo


In [7]:
# one row per variant: the switches are identical across R1..R5
variants = df.drop(columns=["run", "run_name"]).drop_duplicates().set_index("variant")
assert len(variants) == 10, "each variant must collapse to a single switch combination"
variants

,USE_PPO_LAGRANGIAN,CONSTRAINT_TYPES,ENABLE_ACTION_MASK,MASK_RULE_TYPE,ENABLE_NONLINEAR_MAPPING,NONLINEAR_MAPPING_TYPE,POLICY_TYPE
variant,,,,,,,
A1_Correction,False,2,False,basic,False,square,ppo
A2_Penalty,False,3,False,basic,False,square,ppo
A3_Lagrangian,True,3,False,basic,False,square,ppo_lagrangian
B2_WaterMask,True,3,True,basic,False,square,ppo_lagrangian
B3_FullMask,True,3,True,enhanced,False,square,ppo_lagrangian
C2_Square,True,3,True,enhanced,True,square,ppo_lagrangian
C3_Sqrt,True,3,True,enhanced,True,sqrt,ppo_lagrangian
C4_Cubic,True,3,True,enhanced,True,cubic,ppo_lagrangian
C5_Exp,True,3,True,enhanced,True,exp,ppo_lagrangian


### 변형 간 체인 구조

각 변형은 바로 앞 변형에서 스위치 **하나**만 바꿉니다. 아래 코드는 인접한 변형 사이의 차이만 출력합니다.

In [8]:
chain = ["A3_Lagrangian", "A2_Penalty", "A1_Correction", "B2_WaterMask", "B3_FullMask", "C2_Square", "C3_Sqrt", "C4_Cubic", "C5_Exp", "C6_Log"]
base_of = {"A2_Penalty": "A3_Lagrangian", "A1_Correction": "A3_Lagrangian", "B2_WaterMask": "A3_Lagrangian", "B3_FullMask": "B2_WaterMask",
           "C2_Square": "B3_FullMask", "C3_Sqrt": "C2_Square", "C4_Cubic": "C2_Square", "C5_Exp": "C2_Square", "C6_Log": "C2_Square"}
for v in chain[1:]:
    b = base_of[v]
    diff = {k: (variants.loc[b, k], variants.loc[v, k]) for k in SWITCHES if variants.loc[b, k] != variants.loc[v, k]}
    print(f"{v:14s} <- {b:14s} : {diff}")

A2_Penalty     <- A3_Lagrangian  : {'USE_PPO_LAGRANGIAN': (True, False), 'POLICY_TYPE': ('ppo_lagrangian', 'ppo')}
A1_Correction  <- A3_Lagrangian  : {'USE_PPO_LAGRANGIAN': (True, False), 'CONSTRAINT_TYPES': (3, 2), 'POLICY_TYPE': ('ppo_lagrangian', 'ppo')}
B2_WaterMask   <- A3_Lagrangian  : {'ENABLE_ACTION_MASK': (False, True)}
B3_FullMask    <- B2_WaterMask   : {'MASK_RULE_TYPE': ('basic', 'enhanced')}
C2_Square      <- B3_FullMask    : {'ENABLE_NONLINEAR_MAPPING': (False, True)}
C3_Sqrt        <- C2_Square      : {'NONLINEAR_MAPPING_TYPE': ('square', 'sqrt')}
C4_Cubic       <- C2_Square      : {'NONLINEAR_MAPPING_TYPE': ('square', 'cubic')}
C5_Exp         <- C2_Square      : {'NONLINEAR_MAPPING_TYPE': ('square', 'exp')}
C6_Log         <- C2_Square      : {'NONLINEAR_MAPPING_TYPE': ('square', 'log')}


## 4. DI-engine 기본값과의 비교

설정 파일의 `policy` 블록은 DI-engine `PPOPolicy` 의 기본 설정 위에 덮어씌워집니다. 논문에서 바꾼 값과 기본값을 나란히 봅니다.

In [9]:
from ding.policy import PPOPolicy
default = PPOPolicy.default_config()
cmp = []
for section in ("learn", "collect"):
    for k, v in main_config.policy[section].items():
        if k in ("learner", "collector"):
            continue
        cmp.append(dict(section=section, key=k, paper=v, di_engine_default=default[section].get(k, "(n/a)")))
pd.DataFrame(cmp)

[09-20 02:27:29] WARNING  If you want to use numba to speed up segment tree, please install numba first                                              ]8;id=7579995;file:///home/hydro/miniconda3/envs/ppo_rl/lib/python3.10/site-packages/ding/utils/default_helper.py\default_helper.py]8;;\:]8;id=7579996;file:///home/hydro/miniconda3/envs/ppo_rl/lib/python3.10/site-packages/ding/utils/default_helper.py#450\450]8;;\

[09-20 02:27:29] WARNING  Please install pyecharts first, you can install it by running 'pip install pyecharts'                                        ]8;id=7580003;file:///home/hydro/miniconda3/envs/ppo_rl/lib/python3.10/site-packages/ding/utils/memory_helper.py\memory_helper.py]8;;\:]8;id=7580004;file:///home/hydro/miniconda3/envs/ppo_rl/lib/python3.10/site-packages/ding/utils/memory_helper.py#12\12]8;;\

[09-20 02:27:29] WARNING  not found transformer, please install it using: pip install transformers                                               ]8;id=7580011;file:///home/hydro/miniconda3/envs/ppo_rl/lib/python3.10/site-packages/ding/model/template/language_transformer.py\language_transformer.py]8;;\:]8;id=7580012;file:///home/hydro/miniconda3/envs/ppo_rl/lib/python3.10/site-packages/ding/model/template/language_transformer.py#9\9]8;;\

,section,key,paper,di_engine_default
0,learn,debug_print_entropy,False,(n/a)
1,learn,resume_training,False,False
2,learn,epoch_per_collect,10,10
3,learn,batch_size,1460,64
4,learn,learning_rate,0.001,0.0003
5,learn,value_weight,0.6,0.5
6,learn,entropy_weight,0.016,0.0
7,learn,clip_ratio,0.2,0.2
8,learn,adv_norm,True,True
9,learn,value_norm,True,True


## 정리

* 설정 파일은 **학습 코드가 아니라 학습 레시피**입니다. 환경/정책 구현은 비공개이며, 파일은 하이퍼파라미터와 실험 스위치를 기록합니다.
* 10개 변형은 `A3_Lagrangian` 을 기준으로 스위치를 하나씩 바꾼 체인이고, R1–R5 는 이름만 다릅니다(시드는 `--seed` 로 전달).
* 다음 노트북에서는 이 스위치들이 실제로 무엇을 의미하는지 장난감 저수지 환경으로 재현해 봅니다.